In [1]:
import numpy as np
import pandas as pd
import os
def sort_like_merge(df, keys):
    df = df.reset_index(drop=True)
    df["_row"] = np.arange(len(df))
    df = df.sort_values(keys + ["_row"], na_position="first")
    return df.drop(columns="_row").reset_index(drop=True)


def fill_by_group(df, col, by, backward=False):
    g = df.groupby(by, sort=False)[col]
    return g.bfill() if backward else g.ffill()

In [2]:
RAW = r"D:\MIT\replication data collection\data wrds"


def load(name):
    df = pd.read_csv(os.path.join(RAW, name), low_memory=False)
    df.columns = df.columns.str.lower()
    if "gvkey" in df.columns:
        df["gvkey"] = df["gvkey"].astype(str).str.zfill(6)
    return df


names = load("comp_names.csv")
names = names.drop_duplicates("gvkey", keep="last")
adata = load("comp_funda.csv")
compq = load("comp_fundq.csv")
stock_names = load("crspq.stocknames.csv")
link = load("crspq.ccmxpf_lnkhist.csv")
#rename  old:new
link = link.rename(columns={"lpermco": "permco", "lpermno": "permno"})
link = link[link["linktype"].isin(["LC", "LU", "LS"])]
stock_names = stock_names.rename(columns={"secinfostartdt": "namedt", "secinfoenddt": "nameenddt"})
#datadate
#'E'--'NAN'
adata["datadate"] = pd.to_datetime(adata["datadate"])
for col in ["linkdt", "linkenddt"]:
    link[col] = pd.to_datetime(link[col], errors="coerce")
for col in ["namedt", "nameenddt"]:
    stock_names[col] = pd.to_datetime(stock_names[col], errors="coerce")
link["permco"] = link["permco"].astype(float)
stock_names["permco"] = stock_names["permco"].astype(float)

In [3]:
#basic check
print(stock_names.columns.tolist())
print(adata[["indfmt", "datafmt", "consol"]].drop_duplicates())
print(compq.duplicated(["gvkey", "datadate"]).sum())
adata = adata[(adata["indfmt"] == "INDL") & (adata["datafmt"] == "STD") & (adata["consol"] == "C")]
print(compq[["indfmt", "datafmt", "consol"]].drop_duplicates())

['permno', 'hdrprimaryexch', 'nasdissuno', 'hdrsiccd', 'permco', 'namedt', 'nameenddt', 'securitybegdt', 'securityenddt', 'cusip', 'ticker', 'issuernm', 'shareclass', 'usincflg', 'issuertype', 'securitytype', 'securitysubtype', 'sharetype', 'securityactiveflg', 'siccd', 'primaryexch', 'tradingsymbol', 'naics', 'tradingstatusflg']
  indfmt datafmt consol
0   INDL     STD      C
962
  indfmt datafmt consol
0   INDL     STD      C


In [4]:
# ---- Check data integrity ----
print(f"Rows dropped because of missing fyear: {adata['fyear'].isna().mean() * 100:.2f}%")
adata = adata[adata["fyear"].notna()].copy()
adata["fyear"] = adata["fyear"].astype(int)

print(f"Rows dropped because of duplicated (datadate, gvkey): {adata.duplicated(['gvkey', 'datadate']).sum()}")
print(f"Rows dropped because of duplicated (fyear, gvkey): {adata.duplicated(['gvkey', 'fyear']).sum()}")
adata = adata.sort_values(["gvkey", "datadate", "fyear"])
adata = adata.drop_duplicates(["gvkey", "fyear"], keep="last")


Rows dropped because of missing fyear: 0.00%
Rows dropped because of duplicated (datadate, gvkey): 0
Rows dropped because of duplicated (fyear, gvkey): 11


In [5]:

# ---- Merge names table ----
adata = names[["gvkey", "sic", "naics"]].merge(adata, on="gvkey", how="right")

# ---- Fill gaps of missing years ----
span = adata.groupby("gvkey")["fyear"].agg(sdate="min", edate="max").reset_index()
years = pd.DataFrame({"fyear": np.sort(adata["fyear"].unique())})
grid = span.merge(years, how="cross")
grid = grid[(grid["fyear"] >= grid["sdate"]) & (grid["fyear"] <= grid["edate"])]
adata = grid[["gvkey", "fyear"]].merge(adata, on=["gvkey", "fyear"], how="left")
adata = sort_like_merge(adata, ["gvkey", "fyear"])


In [6]:

# ---- Merge Compustat with CRSP (PERMCO) ----
comp_key = adata.loc[adata["datadate"].notna(), ["gvkey", "datadate"]]
linked = comp_key.merge(link, on="gvkey")
linked = linked[linked["datadate"] >= linked["linkdt"]]
linked = linked[(linked["datadate"] <= linked["linkenddt"]) | linked["linkenddt"].isna()]
linked = sort_like_merge(linked, ["gvkey", "datadate", "linkdt", "linkenddt"])
linked = linked.drop_duplicates(["datadate", "gvkey", "permco"])
linked = linked.drop_duplicates(["gvkey", "datadate"])

adata = adata.merge(linked[["gvkey", "datadate", "permco"]], on=["gvkey", "datadate"], how="left")
adata = sort_like_merge(adata, ["gvkey", "datadate"])
print(f"Percentage of compustat rows with valid PERMCO: {adata['permco'].notna().mean() * 100:.2f}")

# ---- Merge with CRSP names table ----
stock_names = stock_names.sort_values(["permco", "nameenddt"], kind="stable")
stock_names = stock_names.drop_duplicates(["permco", "nameenddt"], keep="last")

temp = adata.loc[adata["permco"].notna(), ["permco", "datadate"]].merge(stock_names, on="permco")
temp = temp[(temp["datadate"] >= temp["namedt"]) | temp["namedt"].isna()]
temp = temp[(temp["datadate"] <= temp["nameenddt"]) | temp["nameenddt"].isna()]
temp = temp.sort_values(["permco", "nameenddt"], kind="stable")
temp = temp.drop_duplicates(["permco", "datadate", "siccd"], keep="last")
temp = temp.drop_duplicates(["permco", "datadate"], keep="last")

adata = adata.merge(temp[["permco", "datadate", "siccd", "ticker"]], on=["permco", "datadate"], how="left")
adata = sort_like_merge(adata, ["permco", "datadate"])

# ---- Fill missing industry classification ----
adata["naicsh_filled"] = pd.to_numeric(adata["naicsh"], errors="coerce")
adata["sich_filled"] = pd.to_numeric(adata["sich"], errors="coerce")
print(f"Number of NAICSH missing {adata['naicsh_filled'].isna().mean() * 100:.2f}%.")
print(f"Number of SICH missing {adata['sich_filled'].isna().mean() * 100:.2f}%.")

adata["sich_filled"] = adata["sich_filled"].fillna(pd.to_numeric(adata["siccd"], errors="coerce"))

for col in ["naicsh_filled", "sich_filled"]:
    adata[col] = fill_by_group(adata, col, "gvkey", backward=True)
print(f"Number of NAICSH still missing after backfilling: {adata['naicsh_filled'].isna().mean() * 100:.2f}%.")
print(f"Number of SICH still missing after backfilling: {adata['sich_filled'].isna().mean() * 100:.2f}%.")

adata["naicsh_filled"] = adata["naicsh_filled"].fillna(pd.to_numeric(adata["naics"], errors="coerce"))
# same as the R code: this line fills naicsh_filled (not sich_filled) with the SIC code
no_sic = adata["sich_filled"].isna()
adata.loc[no_sic, "naicsh_filled"] = pd.to_numeric(adata.loc[no_sic, "sic"], errors="coerce")

for col in ["naicsh_filled", "sich_filled"]:
    adata[col] = fill_by_group(adata, col, "gvkey", backward=True)
    adata[col] = fill_by_group(adata, col, "gvkey")

has_permco = adata["permco"].notna()
for col in ["naicsh_filled", "sich_filled"]:
    sub = adata[has_permco]
    adata.loc[has_permco, col] = fill_by_group(sub, col, "permco", backward=True)
    sub = adata[has_permco]
    adata.loc[has_permco, col] = fill_by_group(sub, col, "permco")

print(f"Final share of NAICSH still missing: {adata['naicsh_filled'].isna().mean() * 100:.2f}%.")
print(f"Final share of SICH still missing: {adata['sich_filled'].isna().mean() * 100:.2f}%.")

adata = adata.drop(columns=["sich", "naicsh", "siccd", "naics", "sic"])
adata = adata.rename(columns={"naicsh_filled": "naics", "sich_filled": "sic"})
adata["naics"] = adata["naics"].round().astype("Int64")
adata["sic"] = adata["sic"].round().astype("Int64")


Percentage of compustat rows with valid PERMCO: 64.36
Number of NAICSH missing 22.28%.
Number of SICH missing 30.64%.
Number of NAICSH still missing after backfilling: 18.04%.
Number of SICH still missing after backfilling: 9.83%.
Final share of NAICSH still missing: 1.28%.
Final share of SICH still missing: 7.55%.


In [7]:

out = r"D:\MIT\replication data collection\data\processed\compustat"
os.makedirs(out, exist_ok=True)
adata.to_parquet(os.path.join(out, "compa_clean.parquet"), index=False)
print("Compustat annual has been created successfully.")

Compustat annual has been created successfully.


In [8]:
#--------------------
#compq_clean
#--------------------
OUT = r"D:\MIT\replication data collection\data\processed\compustat"


def to_quarter(s):
    return pd.PeriodIndex(s.astype(str), freq="Q").to_series(index=s.index)


comp = compq.copy()
compa = adata[["gvkey", "fyear", "sic", "naics"]].copy()

needed = ["gvkey", "datadate", "conm", "datafqtr", "datacqtr", "fyearq", "fqtr",
          "atq", "saleq", "cheq", "chq", "prccq", "cshoq", "dlcq", "dlttq",
          "dltisy", "dltry", "dvy", "prstkcy", "sstky"]
missing = [c for c in needed if c not in comp.columns]
print("Missing columns:", missing)

comp["datadate"] = pd.to_datetime(comp["datadate"])
comp["mdate"] = comp["datadate"].dt.year * 100 + comp["datadate"].dt.month

# ---- Clean duplicates and bad values ----
print(f"Rows dropped because of missing datafqtr: {comp['datafqtr'].isna().sum()}")
comp = comp[comp["datafqtr"].notna()]
comp = comp[comp["datacqtr"].notna()]

print(f"Duplicated (gvkey, datadate): {comp.duplicated(['gvkey', 'datadate']).sum()}")
comp = comp.sort_values(["gvkey", "datafqtr", "datadate"])
comp = comp.drop_duplicates(["gvkey", "datadate"], keep="last")
comp = comp.drop_duplicates(["gvkey", "mdate"], keep="last")
comp = comp.drop_duplicates(["gvkey", "datafqtr"], keep="last")

comp = comp[comp["atq"] > 0]
comp = comp[comp["saleq"] > 0].copy()
comp.loc[comp["cheq"] < 0, "cheq"] = np.nan

# ---- Merge industry codes from the annual data ----
comp["fyearq"] = comp["fyearq"].astype("Int64")
compa = compa.rename(columns={"fyear": "fyearq"})
compa["fyearq"] = compa["fyearq"].astype("Int64")
comp = comp.merge(compa, on=["gvkey", "fyearq"], how="left")

# ---- Complete panel: every fiscal quarter between a firm's first and last ----
comp["qdate"] = to_quarter(comp["datafqtr"])
quarters = pd.DataFrame({"qdate": np.sort(comp["qdate"].unique())})
span = comp.groupby("gvkey")["qdate"].agg(first_quarter="min", last_quarter="max").reset_index()
grid = span.merge(quarters, how="cross")
grid = grid[(grid["qdate"] >= grid["first_quarter"]) & (grid["qdate"] <= grid["last_quarter"])]
comp = grid[["gvkey", "qdate"]].merge(comp, on=["gvkey", "qdate"], how="left")

# ---- Year-to-date variables to quarterly ----
comp["fyearq"] = comp["fyearq"].fillna(comp["qdate"].dt.year).astype(int)
comp["fqtr"] = comp["fqtr"].fillna(comp["qdate"].dt.quarter).astype(int)
comp = comp.sort_values(["gvkey", "fyearq", "fqtr"]).reset_index(drop=True)

pvars = ["dltisy", "dltry", "dvy", "prstkcy", "sstky"]
nvars = ["dltisq", "dltrq", "dvq", "prstkcq", "sstkq"]

keys = [comp["gvkey"], comp["fyearq"]]
x = comp[pvars]
filled = x.groupby(keys).ffill()
y = filled - filled.groupby(keys).shift()
y = y.where(~(y.isna() & x.notna()), x)
y = y.mask(x.isna() & (y == 0))
comp[nvars] = y.to_numpy()

# ---- Keep real quarters only, add calendar quarter ----
comp = comp.sort_values(["gvkey", "qdate"]).reset_index(drop=True)
comp = comp[comp["datacqtr"].notna()].copy()
comp["cyqtr"] = to_quarter(comp["datacqtr"])
comp = comp.drop(columns="qdate")

# ---- Fill missing industry codes within each firm ----
for col in ["naics", "sic"]:
    comp[col] = fill_by_group(comp, col, "gvkey", backward=True)
    comp[col] = fill_by_group(comp, col, "gvkey")
print(f"NAICS still missing: {comp['naics'].isna().mean() * 100:.2f}%")

if comp.duplicated(["cyqtr", "gvkey"]).any() or comp.duplicated(["mdate", "gvkey"]).any():
    raise ValueError("Compustat data primary key violated.")

compq_clean = comp
os.makedirs(OUT, exist_ok=True)
compq_clean.to_parquet(os.path.join(OUT, "compq_clean.parquet"), index=False)
print(f"compq_clean: {len(compq_clean):,} rows, {compq_clean['gvkey'].nunique():,} firms")

Missing columns: []
Rows dropped because of missing datafqtr: 0
Duplicated (gvkey, datadate): 0
NAICS still missing: 0.59%
compq_clean: 1,131,862 rows, 28,709 firms


In [9]:
d_q = comp.duplicated(["gvkey", "cyqtr"], keep=False)
d_m = comp.duplicated(["gvkey", "mdate"], keep=False)
print("duplicated calendar quarter:", d_q.sum(), "| duplicated month:", d_m.sum())

cols = ["gvkey", "conm", "datadate", "datafqtr", "datacqtr", "fyearq", "fqtr"]
print(comp.loc[d_q, cols].sort_values(["gvkey", "datadate"]).head(20))

duplicated calendar quarter: 0 | duplicated month: 0
Empty DataFrame
Columns: [gvkey, conm, datadate, datafqtr, datacqtr, fyearq, fqtr]
Index: []


In [10]:
print("rows in compq (raw):", len(compq))
print("rows in comp (now):", len(comp))
print("duplicated gvkey in names:", names["gvkey"].duplicated().sum())
print("duplicated (gvkey, fyear) in adata:", adata.duplicated(["gvkey", "fyear"]).sum())

rows in compq (raw): 1691508
rows in comp (now): 1131862
duplicated gvkey in names: 0
duplicated (gvkey, fyear) in adata: 0


In [11]:
# ======================= 
#annual FIS data (fdata for COVID)
#=======================
FIS_DIR = r"D:\MIT\replication data collection\fis"
FINAL = r"D:\MIT\replication data collection\data\processed\final"


def lead_digits(s, n):
    return s.round().astype("Int64").astype("string").str[:n]


def read_fis(name):
    path = os.path.join(FIS_DIR, name)
    df = pd.read_parquet(path + ".parquet") if os.path.exists(path + ".parquet") else pd.read_csv(path + ".csv")
    df["gvkey"] = df["gvkey"].astype(str).str.split(".").str[0].str.zfill(6)
    return df


fis_annual = read_fis("fis_annual")
fis_quarterly = read_fis("fis_quarterly")
print(fis_annual.columns.tolist())
print(fis_quarterly["year"].head())

# FIS: missing values set to 0, then add 2020Q4 and 2021Q4 as annual observations
fis = fis_annual.fillna({c: 0 for c in fis_annual.select_dtypes("number").columns})
fisq0 = fis_quarterly.fillna({c: 0 for c in fis_quarterly.select_dtypes("number").columns})
fis = pd.concat([fis, fisq0[fisq0["year"].isin(["2020Q4", "2021Q4"])]], ignore_index=True)
fis["year"] = pd.to_numeric(fis["year"].astype(str).replace({"2020Q4": "2020", "2021Q4": "2021"}))
fis = fis.rename(columns={"year": "fyear"}).drop(columns=["conm", "cik"], errors="ignore")
fis["fyear"] = fis["fyear"].astype("Int64")

# Compustat annual sample: positive assets and cash, no financials, utilities or government
comp_data = adata.sort_values(["gvkey", "fyear", "datadate"]).drop_duplicates(["gvkey", "fyear"])
comp_data = comp_data[(comp_data["at"] > 0) & (comp_data["che"] > 0)]
keep = (
    (lead_digits(comp_data["sic"], 1) != "6")
    & (lead_digits(comp_data["naics"], 2) != "52")
    & (lead_digits(comp_data["sic"], 2) != "49")
    & (lead_digits(comp_data["naics"], 2) != "22")
    & (lead_digits(comp_data["sic"], 1) != "9")
)
comp_data = comp_data[keep.fillna(False)].copy()
comp_data["fyear"] = comp_data["fyear"].astype("Int64")
comp_data = comp_data[comp_data["fyear"] >= 2000]

fdata = fis.merge(comp_data[["gvkey", "fyear", "cik"]], on=["fyear", "gvkey"])

fdata["no_cash"] = fdata["total"] - fdata["cash_and_cash_equivalents_total"]
fdata["cash_agg"] = fdata["cash_and_cash_equivalents_total"]
fdata["us_gov_debt_agg"] = fdata["us_treas_agency"] + fdata["others_us_gov_debt"]
fdata["corp_debt_agg"] = fdata["us_corporate"] + fdata["foreign_corporate"]
fdata["equity_agg"] = fdata["us_equity"] + fdata["foreign_equity"]
fdata["other_agg"] = fdata["abs_mbs"] + fdata["foreign_gov_treas"] + fdata["other_debt"] + fdata["others"]

print(f"fdata: {len(fdata):,} firm-years, {fdata['gvkey'].nunique()} firms")



['gvkey', 'cik', 'conm', 'year', 'abs_mbs', 'adjustments', 'cash', 'cash_and_cash_equivalents', 'cash_and_cash_equivalents_total', 'che_difference', 'cp', 'deposits', 'foreign_corporate', 'foreign_equity', 'foreign_gov_treas', 'mmf', 'net_total', 'other_ce', 'other_debt', 'others', 'others_us_gov_debt', 'total', 'us_corporate', 'us_equity', 'us_treas_agency']
0    2020Q1
1    2020Q2
2    2020Q3
3    2020Q4
4    2021Q1
Name: year, dtype: str
fdata: 3,670 firm-years, 199 firms


In [12]:
# =======================
#quarterly data (qdata) 
#=======================
def to_quarter(s):
    return pd.PeriodIndex(s.astype(str), freq="Q").to_series(index=s.index)


def nan_sum(*cols):
    return sum(c.fillna(0) for c in cols)


# ---- Compustat quarterly variables ----
cq = compq_clean.sort_values(["gvkey", "datadate"]).reset_index(drop=True)
g = cq.groupby("gvkey")
cq["atq_lag"] = g["atq"].shift()
cq["cheqg"] = 100 * (cq["cheq"] - g["cheq"].shift()) / cq["atq_lag"]
cq["chqg"] = 100 * (cq["chq"] - g["chq"].shift()) / cq["atq_lag"]

cq["ndiq"] = nan_sum(cq["dltisq"], -cq["dltrq"])
cq["payout"] = nan_sum(cq["dvq"], cq["prstkcq"], -cq["sstkq"])
cq["at_growth"] = cq["atq"] - cq["atq_lag"]
cq["at_growth_ratio"] = cq["at_growth"] / cq["atq_lag"] * 100
cq["nert_lagged_at"] = cq["payout"] * 100 / cq["atq_lag"]
cq["ndi_lagged_at"] = cq["ndiq"] * 100 / cq["atq_lag"]

q1data = pd.DataFrame({
    "gvkey": cq["gvkey"],
    "conm": cq["conm"],
    "datadate": cq["datadate"],
    "datafqtr": to_quarter(cq["datafqtr"]),
    "datacqtr": to_quarter(cq["datacqtr"]),
    "naics": cq["naics"],
    "sic": cq["sic"],
    "at": cq["atq"],
    "CH": cq["chq"],
    "CHE": cq["cheq"],
    "cheqg": cq["cheqg"],
    "chqg": cq["chqg"],
    "atq_lag": cq["atq_lag"],
    "ndiq": cq["ndiq"] / cq["atq_lag"] * 100,
    "at_growth": cq["at_growth"],
    "at_growth_ratio": cq["at_growth_ratio"],
    "nert_lagged_at": cq["nert_lagged_at"],
    "ndi_lagged_at": cq["ndi_lagged_at"],
})
q1data = q1data[q1data["CHE"].notna()]

# ---- Quarterly FIS 2020-2021 (missing values are NOT set to 0 here) ----
fq = fis_quarterly
fisq = pd.DataFrame({
    "gvkey": fq["gvkey"],
    "cik": fq["cik"],
    "datafqtr": to_quarter(fq["year"]),
    "total_fa": fq["total"],
    "cash_agg": fq["cash_and_cash_equivalents_total"],
    "no_cash": fq["total"] - fq["cash_and_cash_equivalents_total"],
    "corp_debt_agg": fq["us_corporate"] + fq["foreign_corporate"],
    "us_gov_debt_agg": fq["us_treas_agency"] + fq["others_us_gov_debt"],
    "equity_agg": fq["us_equity"] + fq["foreign_equity"],
    "other_agg": fq["abs_mbs"] + fq["foreign_gov_treas"] + fq["other_debt"] + fq["others"],
})
fisq["fyear"] = fisq["datafqtr"].dt.year

# ---- Annual FIS as fiscal Q4, stacked with quarterly FIS ----
fisa = fdata[["gvkey", "cik", "fyear", "total", "cash_agg", "no_cash", "corp_debt_agg",
              "us_gov_debt_agg", "equity_agg", "other_agg"]].rename(columns={"total": "total_fa"})
fisa["datafqtr"] = pd.PeriodIndex(fisa["fyear"].astype(int).astype(str) + "Q4", freq="Q")

fis_all = pd.concat([fisa, fisq], ignore_index=True)
fis_all = fis_all.drop_duplicates(["gvkey", "datafqtr"])
fis_all["cik"] = fis_all["cik"].astype(str)

prev = fis_all[["gvkey", "datafqtr", "cash_agg", "total_fa"]].rename(
    columns={"cash_agg": "ch_minus_1", "total_fa": "fa_minus_1"})
prev["datafqtr"] = prev["datafqtr"] + 1
fis_all = fis_all.merge(prev, on=["gvkey", "datafqtr"], how="left")

# ---- Merge with Compustat: firms with FIS data in 2019 ----
sgvkeys = fis_all.loc[fis_all["fyear"] == 2019, "gvkey"].unique()
qdata = fis_all.merge(q1data[q1data["gvkey"].isin(sgvkeys)], on=["gvkey", "datafqtr"])
qdata["fag"] = 100 * (qdata["total_fa"] - qdata["fa_minus_1"]) / qdata["atq_lag"]
qdata["cash_likeg"] = 100 * (qdata["cash_agg"] - qdata["ch_minus_1"]) / qdata["atq_lag"]
qdata = qdata[qdata["total_fa"].notna()]
qdata = qdata.sort_values(["gvkey", "datafqtr"]).reset_index(drop=True)

os.makedirs(FINAL, exist_ok=True)
qdata.to_parquet(os.path.join(FINAL, "qdata.parquet"), index=False)
print(f"qdata: {len(qdata):,} rows, {qdata['gvkey'].nunique()} firms")

qdata: 3,687 rows, 156 firms


In [13]:
# Trace where firms in the FIS sample drop out of the pipeline:
# count FIS firms that reach fdata and qdata, then show the 2019 Compustat
# records (assets, cash, industry codes) of firms that never reach qdata
fis19 = set(fis_annual.loc[fis_annual["year"].astype(str) == "2019", "gvkey"])
print("FIS firms in 2019:", len(fis19))
print("  in fdata:", len(fis19 & set(fdata["gvkey"])))
print("  in qdata:", len(fis19 & set(qdata["gvkey"])))

lost = fis19 - set(qdata["gvkey"])
print(adata.loc[adata["gvkey"].isin(lost) & (adata["fyear"] == 2019),
                ["gvkey", "conm", "at", "che", "sic", "naics"]].to_string())

FIS firms in 2019: 200
  in fdata: 199
  in qdata: 156
         gvkey                         conm       at      che   sic   naics
287293  001300  HONEYWELL INTERNATIONAL INC  58679.0  10416.0  9997  999977
394583  024616          PAYPAL HOLDINGS INC  51333.0  10761.0  7389  522320


In [14]:
# For firms missing from qdata, check whether they still exist in Compustat:
# a last year before 2019 means the firm was delisted, acquired, or merged
# before the COVID sample period, so its exclusion is expected
chk = adata[adata["gvkey"].isin(lost)].groupby("gvkey").agg(
    conm=("conm", "last"), first_year=("fyear", "min"), last_year=("fyear", "max"))
print(chk.sort_values("last_year").to_string())
print("not in Compustat annual at all:", sorted(lost - set(adata["gvkey"])))

                                conm  first_year  last_year
gvkey                                                      
006127                    ENRON CORP        1990       2000
010482                    TEXACO INC        1990       2000
010301                       TRW INC        1990       2001
003282          COMPAQ COMPUTER CORP        1990       2001
114303                    CONOCO INC        1993       2001
007536                PHARMACIA CORP        1990       2002
134845    AT&T WIRELESS SERVICES INC        1997       2003
005134          GEORGIA-PACIFIC CORP        1990       2004
024870     NEXTEL COMMUNICATIONS INC        1990       2004
009563            SEARS ROEBUCK & CO        1990       2004
001581                     AT&T CORP        1990       2004
001164                       MCI INC        1990       2004
115872   FOX ENTERTAINMENT GROUP INC        1995       2004
012484        ADELPHIA COMMUN  -CL A        1989       2005
001240               ALBERTSON'S INC    